In [1]:
import sys
sys.path.append("..")          # ajoute la racine du projet

import itertools
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import least_squares

from src.validation.splits import choisir_caches
from src.validation.residue import rmse
from src.curves.data import charger_courbe
from src.curves.svensson import sv_rate, _residus, BORNES_SV
from src.curves.evaluation import ajuster_historique

NOMS_SV = ["beta0", "beta1", "beta2", "beta3", "tau1", "tau2"]

In [2]:
chemin_yield="../data/raw/yield_curve.csv"
chemin_param="../data/raw/param.csv"
courbe=charger_courbe(chemin_yield)

bornes = [0, 2, 5, 10, 20, 30]
maturites = pd.Series(courbe.columns, index=courbe.columns)   # l'index EST la maturité
caches, ancrage = choisir_caches(maturites, bornes)
masque_cache = courbe.columns.isin(caches)

## 1. Nouvelle grille : synthèse de l'historique Svensson (`hist_sv.csv`)

In [3]:
# synthèse des résultats (historiques NS et Svensson, grille large)
from src.curves.synthese import synthese_historique, rmse_par_annee

ns = pd.read_csv("../data/processed/hist_ns.csv", index_col=0, parse_dates=True)
sv = pd.read_csv("../data/processed/hist_sv.csv", index_col=0, parse_dates=True)   # grille large (56 départs)

tableau = pd.concat({"NS": synthese_historique(ns, ["tau"]),
                     "SV": synthese_historique(sv, ["tau1", "tau2"])},
                    axis=1, sort=False)      # sort=False : garde l'ordre des lignes
print(tableau.to_string())
print(rmse_par_annee(ns, sv))

                                            NS          SV
dates                                     5662        5662
RMSE moyenne (pb)                     3.423589    0.000321
RMSE médiane (pb)                     2.274705    0.000029
RMSE quantile 95 % (pb)              10.754095     0.00015
RMSE max (pb)                        16.116596    0.048178
date du max                         2012-03-29  2018-02-02
dates RMSE > 0.05 pb                      5388           0
dates RMSE > 5 pb                         1315           0
dates RMSE > 10 pb                         455           0
succes = False                               0          54
replis                                       0          34
dates |beta| > 20 % (non lisible)           55        1359
tau > 20 ans                                83         NaN
tau1 > 20 ans                              NaN           0
tau2 > 20 ans                              NaN           0
tau1/tau2 entre 0,77 et 1,3                NaN        25

## 2. Ancienne grille (15 départs) contre grille large (56 départs)

`hist_sv_grille15.csv` est reconstruit une seule fois avec `fit_sv_v1` (copie de `fit_sv` avant l'élargissement de la grille). La grille large a été choisie après avoir vu l'échec de 42 dates : correctif postérieur, à écrire comme tel dans le rapport.

In [4]:
def fit_sv_v1(T, taux_obs, x_prev=None, seuil_repli_pb=0.05):
    """Copie de fit_sv avant l'élargissement de la grille, uniquement pour ce diagnostic."""
    resultats = []
    if x_prev is not None:
        res = least_squares(_residus, list(x_prev), args=(T, taux_obs), bounds=BORNES_SV)
        if np.sqrt(np.mean(res.fun ** 2)) * 1e4 <= seuil_repli_pb:
            res.repli = False
            return res
        resultats.append(res)
    beta0 = taux_obs[T > 20].mean()
    beta1 = taux_obs[T < 1].mean() - beta0
    departs = [[beta0, beta1, 0.0, 0.0, t1, t2]
               for t1, t2 in itertools.product(np.geomspace(0.5, 8, 4), np.geomspace(3, 30, 4))
               if not np.isclose(t1, t2)]
    resultats += [least_squares(_residus, x0, args=(T, taux_obs), bounds=BORNES_SV) for x0 in departs]
    res = min(resultats, key=lambda r: r.cost)
    res.repli = x_prev is not None
    return res


# Ancienne grille : 2 à 3 minutes la première fois, puis lue depuis le fichier
chemin_g15 = Path("../data/processed/hist_sv_grille15.csv")
if chemin_g15.exists():
    sv_g15 = pd.read_csv(chemin_g15, index_col=0, parse_dates=True)
else:
    sv_g15 = ajuster_historique(courbe, fit_sv_v1, sv_rate, NOMS_SV)
    sv_g15.to_csv(chemin_g15)

sv_g56 = sv        # grille large : même fichier que sv (hist_sv.csv)

print((sv_g15["rmse_pb"] > 0.05).sum(), sv_g15["repli"].sum())   # attendu : 42 et 74
print((sv_g56["rmse_pb"] > 0.05).sum(), sv_g56["repli"].sum())   # nouveau : 0 et le nombre de replis

500/5662 dates (2006-08-14)
1000/5662 dates (2008-07-30)
1500/5662 dates (2010-07-14)
2000/5662 dates (2012-06-21)
2500/5662 dates (2014-06-09)
3000/5662 dates (2016-05-25)
3500/5662 dates (2018-05-09)
4000/5662 dates (2020-04-29)
4500/5662 dates (2022-04-11)
5000/5662 dates (2024-03-22)
5500/5662 dates (2026-03-11)
42 74
0 34


In [5]:
dates_echec = sv_g15.index[sv_g15["rmse_pb"] > 0.05]
comparaison = pd.DataFrame({"grille15_pb": sv_g15.loc[dates_echec, "rmse_pb"],
                            "grille56_pb": sv_g56.loc[dates_echec, "rmse_pb"]})
print(comparaison.sort_values("grille15_pb", ascending=False).head(10))
print((comparaison["grille56_pb"] <= 0.05).sum(), "dates sur", len(comparaison), "retrouvées")

            grille15_pb  grille56_pb
DATE                                
2008-10-08     5.081320     0.000028
2008-10-07     4.837555     0.000029
2008-10-13     0.279360     0.000027
2016-04-25     0.264644     0.000027
2016-04-22     0.260218     0.000028
2016-04-21     0.258301     0.000030
2016-04-20     0.258086     0.000028
2016-04-15     0.237881     0.000028
2008-10-09     0.236607     0.000028
2016-04-19     0.236381     0.000029
42 dates sur 42 retrouvées


## 3. Ordre de TAU1 et TAU2 : paramètres BCE contre notre ajustement

In [6]:
def ordre_tau(params, tol=1e-6):
    """Classe chaque date selon l'ordre de TAU1 et TAU2 dans les paramètres BCE.

    params : DataFrame (index = dates) avec les colonnes "TAU1" et "TAU2".
    tol : écart (en années) en dessous duquel TAU1 et TAU2 sont considérés égaux.
    Retourne une Series indexée par date : "tau1<tau2", "tau1>tau2" ou "egal".
    """
    ecart = params["TAU1"] - params["TAU2"]
    return pd.Series(np.where(ecart.abs() <= tol, "egal",
                              np.where(ecart < 0, "tau1<tau2", "tau1>tau2")),
                     index=params.index, name="ordre_tau")

In [7]:
#on regarde quand les paramètres de la BCE changent de région 
from src.curves.data import charger_parametres

params = charger_parametres("../data/raw/param.csv")
ordre = ordre_tau(params)

print(ordre.value_counts())                          # nombre de dates dans chaque cas
print(pd.crosstab(ordre.index.year, ordre))          # répartition par année
print(params[["TAU1", "TAU2"]].agg(["min", "max"]))  # plages observées de TAU1 et TAU2

hors_ordre = ordre[ordre != "tau1<tau2"]
print(hors_ordre.index.min(), hors_ordre.index.max(), len(hors_ordre))

print(params.index.equals(sv.index))            # mêmes dates que notre historique, avant toute comparaison

ordre_tau
tau1<tau2    3195
tau1>tau2    2467
Name: count, dtype: int64
ordre_tau  tau1<tau2  tau1>tau2
row_0                          
2004               0         85
2005              61        196
2006             255          0
2007             255          0
2008             228         28
2009               0        256
2010               0        258
2011              36        221
2012               0        256
2013               0        255
2014               0        255
2015             241         13
2016             257          0
2017             255          0
2018              14        239
2019             251          2
2020             245         10
2021              27        230
2022             144        113
2023             205         50
2024             255          0
2025             255          0
2026             211          0
          TAU1       TAU2
min   0.250000   0.250000
max  17.109755  16.910175
2004-09-06 00:00:00 2023-03-10 00:00:00 2467
True


In [8]:
#observation si on a les meme régions où tau1<tau2 /tau1>tau2 que dans la BCE
ordre_sv = ordre_tau(sv.rename(columns={"tau1": "TAU1", "tau2": "TAU2"}))
meme_region = (ordre_sv == ordre)          # même index, vérifié (True plus haut)
print(f"Même région {meme_region.mean()}")                    # proportion de dates dans la même région
print(pd.crosstab(ordre, ordre_sv))        # matrice de concordance
print(meme_region.groupby(meme_region.index.year).mean().round(2))   # par année

Même région 0.7767573295655246
ordre_tau  egal  tau1<tau2  tau1>tau2
ordre_tau                            
tau1<tau2     2       2111       1082
tau1>tau2     2        178       2287
DATE
2004    1.00
2005    0.99
2006    1.00
2007    1.00
2008    0.52
2009    1.00
2010    1.00
2011    0.92
2012    1.00
2013    1.00
2014    1.00
2015    1.00
2016    0.32
2017    0.00
2018    0.46
2019    0.01
2020    0.18
2021    0.89
2022    0.75
2023    1.00
2024    1.00
2025    1.00
2026    1.00
Name: ordre_tau, dtype: float64


## 4. Écart entre notre courbe et la courbe reconstruite avec les paramètres BCE

In [9]:
T = courbe.columns.to_numpy(dtype=float)
noms_bce = ["BETA0", "BETA1", "BETA2", "BETA3", "TAU1", "TAU2"]
noms_sv = ["beta0", "beta1", "beta2", "beta3", "tau1", "tau2"]

#on recalcule les taux avec les paramètres
courbe_bce = pd.DataFrame(sv_rate(T, *[params[c].to_numpy()[:, None] for c in noms_bce]),
                          index=params.index, columns=T)
courbe_sv = pd.DataFrame(sv_rate(T, *[sv[c].to_numpy()[:, None] for c in noms_sv]),
                           index=sv.index, columns=T)

ecart_pb = (courbe_sv - courbe_bce).abs().max(axis=1) * 1e4     # écart max entre les deux courbes, par date
desaccord = ~meme_region
print(ecart_pb[desaccord].describe())      # courbes différentes ou non sur les 669 dates ?
print(ecart_pb[~desaccord].describe())

count    1264.000000
mean        0.001471
std         0.011224
min         0.000011
25%         0.000052
50%         0.000082
75%         0.000149
max         0.137973
dtype: float64
count    4398.000000
mean        0.002325
std         0.085192
min         0.000005
25%         0.000036
50%         0.000053
75%         0.000077
max         5.614997
dtype: float64


In [10]:
#focus sur les dates où l'écart >0.05
suspects = ecart_pb[ecart_pb > 0.05].sort_values(ascending=False)
print(len(suspects))
print(suspects.head(10))

d = suspects.index[0]
print((courbe.loc[d] - courbe_bce.loc[d]).abs().max() * 1e4)    # écart paramètres BCE / courbe observée
print((courbe.loc[d] - courbe_sv.loc[d]).abs().max() * 1e4)   # écart nos paramètres / courbe observée

39
DATE
2022-06-23    5.614997
2025-03-13    0.252068
2025-06-05    0.239809
2025-03-17    0.230472
2024-10-02    0.148893
2024-11-07    0.145272
2018-02-02    0.137973
2020-01-02    0.137316
2022-06-10    0.134149
2020-01-09    0.131204
dtype: float64
5.61504100239401
5.406156199788681e-05


In [11]:
# origine des écarts de courbe avec la BCE : paramètres BCE ou notre ajustement ?
ecart_bce_obs = (courbe.loc[suspects.index] - courbe_bce.loc[suspects.index]).abs().max(axis=1) * 1e4
ecart_nous_obs = (courbe.loc[suspects.index] - courbe_sv.loc[suspects.index]).abs().max(axis=1) * 1e4
cause = pd.Series(np.where(ecart_bce_obs > ecart_nous_obs, "paramètres BCE", "notre ajustement"),
                  index=suspects.index)
print(cause.value_counts())
print(pd.DataFrame({"ecart_BCE_obs_pb": ecart_bce_obs, "ecart_nous_obs_pb": ecart_nous_obs,
                    "cause": cause}).sort_values("ecart_BCE_obs_pb", ascending=False).head(10))

notre ajustement    38
paramètres BCE       1
Name: count, dtype: int64
            ecart_BCE_obs_pb  ecart_nous_obs_pb             cause
DATE                                                             
2022-06-23          5.615041           0.000054    paramètres BCE
2022-06-13          0.000345           0.098439  notre ajustement
2022-06-10          0.000245           0.133978  notre ajustement
2018-02-06          0.000197           0.091971  notre ajustement
2018-02-02          0.000188           0.137899  notre ajustement
2024-10-17          0.000144           0.078482  notre ajustement
2018-01-29          0.000136           0.055086  notre ajustement
2024-10-02          0.000131           0.148979  notre ajustement
2024-10-14          0.000129           0.108526  notre ajustement
2018-02-05          0.000129           0.103977  notre ajustement
